In [1]:
import pandas as pd

df = pd.read_csv('2025_Airbnb_NYC_listings.csv')

##### property_type 전처리

In [2]:
#기본정보 확인
print(df['property_type'].describe())

count                  22308
unique                    57
top       Entire rental unit
freq                    9648
Name: property_type, dtype: object


In [3]:
#결측치 개수 확인
nan_count = df['property_type'].isna().sum()
print(f"결측치 개수: {nan_count}개")

결측치 개수: 0개


In [4]:
#숙소 유형별로 얼마나 있는지 확인
print("숙소 유형별 빈도수 (Top 10)")
property_counts = df['property_type'].value_counts()
print(property_counts.head(10))

숙소 유형별 빈도수 (Top 10)
property_type
Entire rental unit             9648
Private room in rental unit    5189
Private room in home           1886
Entire home                    1059
Room in hotel                   814
Entire condo                    691
Private room in townhouse       622
Private room in condo           341
Entire townhouse                336
Entire loft                     291
Name: count, dtype: int64


In [5]:
#결측치가 없으니 통계과정에서 노이즈가 될 수 있는 데이터가 뭐가 있을까 고민해봄
#llm에게 물어보니까 -> 데이터가 너무 작으면 분석에 방해가 될 수 있다고 함.
#그래서 전체 데이터의 1% 미만인 데이터들을 뽑아봄

prop_counts = df['property_type'].value_counts()
prop_pct = df['property_type'].value_counts(normalize=True) * 100

# 1% 미만 비중을 차지하는 데이터
rare = pd.DataFrame({'Count': prop_counts, 'Percentage': prop_pct})
rare_types = rare[rare['Percentage'] < 1.0]

#편의상 1%미만의 데이터를 희귀 숙소라고 지칭
print("\n희귀 숙소 분석")
print(f"종류 수: {len(rare_types)}개")
print(f"전체 데이터에서 차지하는 비중: {rare_types['Percentage'].sum():.2f}%")
print("가장 희소한 숙소:", rare_types.tail(5).index.tolist())
#목장, 탑, 돔 등 -> 숙소보다는 체험?같은걸 하기 위해 방문하는 곳인듯 함


희귀 숙소 분석
종류 수: 46개
전체 데이터에서 차지하는 비중: 5.20%
가장 희소한 숙소: ['Private room in ranch', 'Shared room in casa particular', 'Private room in cottage', 'Private room in tower', 'Private room in dome']


In [6]:
#이러한 데이터가 있다는 것을 생각하고, 소희.찬희님이 해주신 것처럼 희귀숙소를 other로 묶어서 분류하면 될 듯 

##### room_type 전처리

In [7]:
print(df['room_type'].value_counts(dropna=False))
#데이터 타입 : 범주형

room_type
Entire home/apt    12664
Private room        9186
Hotel room           372
Shared room           86
Name: count, dtype: int64


In [8]:
#비어있는(NaN) 행이 있는지 확인
room_nan = df['room_type'].isna().sum()
print(f"\nroom_type 결측치: {room_nan}개")


room_type 결측치: 0개


##### propety_type과 room_type 비교전처리

In [9]:
#room_type은 entire인데 property_type이 'Private room'인 경우가 있을지?
#'room_type'이 Entire home/apt 인 데이터만 따로 추출
entire_homes = df[df['room_type'] == 'Entire home/apt']

#property_type에 'Private'이라는 글자가 들어간 행이 있는지 확인
#llm에게 물어봄 -> 대소문자 구분 없이 찾기 위해서 case=False 사용
mismatch = entire_homes[entire_homes['property_type'].str.contains('Private', case=False, na=False)]

print(f"전체 대여(Entire) 중 'Private' 문구가 포함된 데이터: {len(mismatch)}건")
#room_type이 entire인데 property_type이 'Private'인 경우는 없음 -> 데이터가 일관적임

전체 대여(Entire) 중 'Private' 문구가 포함된 데이터: 0건


##### accomodates 전처리

In [10]:
print(df['accommodates'].describe())
#데이터 타입 : 실수형

count    22308.000000
mean         2.919446
std          2.073278
min          1.000000
25%          2.000000
50%          2.000000
75%          4.000000
max         16.000000
Name: accommodates, dtype: float64


In [11]:
#결측치 및 0인 데이터 확인
nan = df['accommodates'].isna().sum()
zero = (df['accommodates'] == 0).sum()
print(f"\n결측치 개수: {nan}개")
print(f"0인 데이터: {zero}개")


결측치 개수: 0개
0인 데이터: 0개


In [12]:
#이상치 확인 (너무 큰 숫자 뽑아내기)
quantile_99 = df['accommodates'].quantile(0.99)
over_10 = df[df['accommodates'] > 10]

print(f"\n이상치 분석")
print(f"상위 1% 기준: {quantile_99}명")
print(f"10명 초과 수용 숙소 개수: {len(over_10)}개")
#보통 뉴욕 아파트에서 10명 이상 수용은 드문 경우라고 함


이상치 분석
상위 1% 기준: 11.930000000000291명
10명 초과 수용 숙소 개수: 246개


뉴욕 기준의 법적/안전적 근거
뉴욕 시는 소방 안전 및 주거법상 한 공간에 머물 수 있는 인원을 엄격하게 제한함
방 하나에 8명 이상이 있는 것은 뉴욕 주거법상 불법일 확률이 매우 높음

즉, 데이터에 10명, 12명이라고 적혀 있다면 호스트가 검색 노출을 위해 수용 인원으로 거짓말을 쳤거나, 정말 특이한 형태의 불법 숙소일 수 있수도 있음

In [13]:
#room_type과의 논리적 모순 체크
#'Shared room'인데 수용 인원이 10명이다? (도미토리(게하)일 수도 있지만 체크가 필요하다고 생각했음)
shared_large = df[(df['room_type'] == 'Shared room') & (df['accommodates'] > 8)]
print(f"\nShared인데 8명 초과인 숙소: {len(shared_large)}개")
if not shared_large.empty:
    print(shared_large[['property_type', 'accommodates']].head())


Shared인데 8명 초과인 숙소: 0개


In [14]:
# Shared room만 필터링해서 수용 인원의 분포 확인
shared_rooms = df[df['room_type'] == 'Shared room']

print("Shared room 수용 인원 분위수")
print(shared_rooms['accommodates'].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
#shared room 데이터는 깔끔해서 그대로 가져가도 될듯함

Shared room 수용 인원 분위수
0.25    1.00
0.50    2.00
0.75    2.00
0.90    2.50
0.95    4.00
0.99    5.15
Name: accommodates, dtype: float64


##### bathrooms를 정수형으로 바꾸고, amenities 에서 새로운 파생 컬럼 만들어보기

In [15]:
print(df['bathrooms'].describe())

count    22302.000000
mean         1.192897
std          0.556670
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         15.500000
Name: bathrooms, dtype: float64


In [16]:
df['bathrooms'] = df['bathrooms'].fillna(0).astype(float)
df['bedrooms'] = df['bedrooms'].fillna(0).astype(float)
df['beds'] = df['beds'].fillna(0).astype(float)
# 결측치를 채워준 이유 -> 숙소에 욕실이 없는 경우도 있을 수 있기 때문에, 결측치를 0으로 채워주는 것이 논리적으로 맞다고 판단함